In [1]:
import math
import numpy as np
import pandas as pd
import QuantLib as ql


In [2]:
# =============================================================================
# 0.  QuantLib setup
# =============================================================================
trade_date = ql.Date(5, 11, 2025)
calendar   = ql.TARGET()
spot_date  = calendar.advance(trade_date, 2, ql.Days)
dc_curve   = ql.Actual365Fixed()
dc_deposit = ql.Actual360()

print(f"Trade date : {trade_date}   Spot date : {spot_date}")

Trade date : November 5th, 2025   Spot date : November 7th, 2025


In [3]:
# =============================================================================
# 1.  Load the interpolated term structure
# =============================================================================
ts = pd.read_excel("Interp_term_structure.xlsx")
assert "Days" in ts.columns and "DF" in ts.columns

_days_arr = ts["Days"].to_numpy(dtype=float)
_df_arr   = ts["DF"].to_numpy(dtype=float)

def days_from_spot(d: ql.Date) -> int:
    return d.serialNumber() - spot_date.serialNumber()

def get_df(d: ql.Date) -> float:
    """Risk-free discount factor P(spot, d)."""
    n = days_from_spot(d)
    if n <= 0:
        return 1.0
    return float(np.interp(n, _days_arr, _df_arr))

def get_ttm(d: ql.Date) -> float:
    """Act/365 year fraction from spot to d."""
    return dc_curve.yearFraction(spot_date, d)

print(f"Curve loaded: {len(ts)} rows  "
      f"| {ts['Days'].min():.0f} – {ts['Days'].max():.0f} days")


Curve loaded: 18263 rows  | 0 – 18262 days


In [4]:

# =============================================================================
# 2.  Bond constants
# =============================================================================
N          = 1_000.0
ALPHA      = 0.25
CAP_RATE   = 0.0545
FLOOR_RATE = 0.0000
PART       = 1.60
L_STAR     = CAP_RATE / PART

L6 = 0.02029
R6 = min(CAP_RATE, max(FLOOR_RATE, PART * L6))
C6 = N * R6 * ALPHA

# =============================================================================
# 3.  Payment schedule and period start dates (periods 6-40)
# =============================================================================
def qldate(d, m, y): return ql.Date(d, m, y)

SCHEDULE = {
     6: qldate(12, 12, 2025),   7: qldate(12,  3, 2026),
     8: qldate(12,  6, 2026),   9: qldate(14,  9, 2026),
    10: qldate(14, 12, 2026),  11: qldate(12,  3, 2027),
    12: qldate(14,  6, 2027),  13: qldate(13,  9, 2027),
    14: qldate(13, 12, 2027),  15: qldate(13,  3, 2028),
    16: qldate(12,  6, 2028),  17: qldate(12,  9, 2028),
    18: qldate(12, 12, 2028),  19: qldate(12,  3, 2029),
    20: qldate(12,  6, 2029),  21: qldate(12,  9, 2029),
    22: qldate(12, 12, 2029),  23: qldate(12,  3, 2030),
    24: qldate(12,  6, 2030),  25: qldate(12,  9, 2030),
    26: qldate(12, 12, 2030),  27: qldate(12,  3, 2031),
    28: qldate(12,  6, 2031),  29: qldate(12,  9, 2031),
    30: qldate(12, 12, 2031),  31: qldate(12,  3, 2032),
    32: qldate(14,  6, 2032),  33: qldate(13,  9, 2032),
    34: qldate(13, 12, 2032),  35: qldate(14,  3, 2033),
    36: qldate(13,  6, 2033),  37: qldate(12,  9, 2033),
    38: qldate(12, 12, 2033),  39: qldate(13,  3, 2034),
    40: qldate(12,  6, 2034),
}

# Period start = previous period's payment date
PERIOD_STARTS = {
     6: qldate(12,  9, 2025),   7: qldate(12, 12, 2025),
     8: qldate(12,  3, 2026),   9: qldate(12,  6, 2026),
    10: qldate(14,  9, 2026),  11: qldate(14, 12, 2026),
    12: qldate(12,  3, 2027),  13: qldate(14,  6, 2027),
    14: qldate(13,  9, 2027),  15: qldate(13, 12, 2027),
    16: qldate(13,  3, 2028),  17: qldate(12,  6, 2028),
    18: qldate(12,  9, 2028),  19: qldate(12, 12, 2028),
    20: qldate(12,  3, 2029),  21: qldate(12,  6, 2029),
    22: qldate(12,  9, 2029),  23: qldate(12, 12, 2029),
    24: qldate(12,  3, 2030),  25: qldate(12,  6, 2030),
    26: qldate(12,  9, 2030),  27: qldate(12, 12, 2030),
    28: qldate(12,  3, 2031),  29: qldate(12,  6, 2031),
    30: qldate(12,  9, 2031),  31: qldate(12, 12, 2031),
    32: qldate(12,  3, 2032),  33: qldate(14,  6, 2032),
    34: qldate(13,  9, 2032),  35: qldate(13, 12, 2032),
    36: qldate(14,  3, 2033),  37: qldate(13,  6, 2033),
    38: qldate(12,  9, 2033),  39: qldate(12, 12, 2033),
    40: qldate(13,  3, 2034),
}

MATURITY = SCHEDULE[40]

# UniCredit 5Y EUR CDS, trade date 05-Nov-2025  (UNIC5YEUAM=R)
CDS_SPOT_BP = 38.78

# =============================================================================
# 4.  Forward EURIBOR and structured coupon
# =============================================================================
def forward_euribor(start: ql.Date, end: ql.Date) -> float:
    """Implied 3m forward EURIBOR: f = (1/alpha) * (P(start)/P(end) - 1)."""
    alpha = dc_deposit.yearFraction(start, end)   # Act/360
    return (1.0 / alpha) * (get_df(start) / get_df(end) - 1.0)

def structured_coupon_rate(fwd: float) -> float:
    return min(CAP_RATE, max(FLOOR_RATE, PART * fwd))

In [5]:

# =============================================================================
# 5.  Build full exposure schedule (coupons 6-40 + principal)
# =============================================================================
exp_rows = []

for period, pay_date in SCHEDULE.items():
    tau_i = get_ttm(pay_date)
    df_i  = get_df(pay_date)

    if period == 6:
        fwd         = L6
        coup_rate   = R6
        coupon_eur  = C6
    else:
        fwd         = forward_euribor(PERIOD_STARTS[period], pay_date)
        coup_rate   = structured_coupon_rate(fwd)
        coupon_eur  = N * coup_rate * ALPHA

    exp_rows.append({
        "Period"      : period,
        "Payment Date": pay_date.ISO(),
        "tau_i"       : tau_i,
        "Fwd EURIBOR" : fwd,
        "Coupon Rate" : coup_rate,
        "Exposure"    : coupon_eur,
        "Z_rf"        : df_i,
    })

# Principal row
tau_40 = get_ttm(MATURITY)
df_40  = get_df(MATURITY)
exp_rows.append({
    "Period"      : "Principal",
    "Payment Date": MATURITY.ISO(),
    "tau_i"       : tau_40,
    "Fwd EURIBOR" : np.nan,
    "Coupon Rate" : np.nan,
    "Exposure"    : N,
    "Z_rf"        : df_40,
})

exp_df = pd.DataFrame(exp_rows)

# Risk-free NPV
rf_npv = 1102.2711  # Q5 BASE_GROSS_PRICE

# Accrued interest (30/360): period 6 start 12 Sep 2025, trade 5 Nov 2025
_days_acc = (2025-2025)*360 + (11-9)*30 + (5-12)   # 53
AI        = N * R6 * (_days_acc / 360)              # 4.779

print(f"\nRisk-free NPV (gross) : EUR {rf_npv:.2f}")
print(f"Accrued interest      : EUR {AI:.3f}")
print(f"Risk-free clean price : EUR {rf_npv - AI:.2f}  ({(rf_npv-AI)/10:.2f}%)")

# =============================================================================
# 6.  CVA function  (Fusai Risky Bonds eq. 6, rearranged)
# =============================================================================
def compute_cva(cds_spread: float,
                recovery:   float = 0.40) -> tuple:
    """
    CVA = (1-R) * N * sum_i [ Q(T_{i-1}) - Q(T_i) ] * P(t, T_i)

    where  Q(T) = exp(-lambda * T),  lambda = cds_spread / (1 - recovery)

    Returns (cva_total, cva_coupons, cva_principal, detail_df).
    The split into cva_coupons / cva_principal follows the same interval
    attribution: intervals ending at a coupon date vs the maturity date.
    """
    lam  = cds_spread / (1.0 - recovery)
    lgd  = 1.0 - recovery

    # Build ordered list of (tau_i, df_i, is_principal)
    schedule_items = []
    for _, r in exp_df.iterrows():
        schedule_items.append((r["tau_i"], r["Z_rf"],
                               r["Period"] == "Principal",
                               r["Period"], r["Payment Date"]))

    cva_coupons   = 0.0
    cva_principal = 0.0
    rows          = []
    Q_prev        = 1.0

    for tau_i, zrf_i, is_principal, period, pay_date in schedule_items:
        Q_i   = math.exp(-lam * tau_i)
        dQ    = Q_prev - Q_i                          # marginal default prob in (T_{i-1}, T_i]
        cva_i = lgd * N * dQ * zrf_i                  # Fusai eq. (6) recovery term

        if is_principal:
            cva_principal += cva_i
        else:
            cva_coupons += cva_i

        rows.append({
            "Period"      : period,
            "Payment Date": pay_date,
            "tau_i"       : round(tau_i,   4),
            "Z_rf"        : round(zrf_i,   6),
            "Q(T_{i-1})"  : round(Q_prev,  6),
            "Q(T_i)"      : round(Q_i,     6),
            "dQ"          : round(dQ,      6),
            "CVA_i EUR"   : round(cva_i,   4),
        })
        Q_prev = Q_i

    cva_total = cva_coupons + cva_principal
    return cva_total, cva_coupons, cva_principal, pd.DataFrame(rows)

# =============================================================================
# 7.  Base-case results  (CDS = 38.78 bp, R = 40%)
# =============================================================================
RECOVERY_BASE = 0.40

cva_total, cva_coupons, cva_principal, cva_detail = \
    compute_cva(CDS_SPOT_BP / 1e4, RECOVERY_BASE)

risky_npv   = rf_npv - cva_total
risky_clean = risky_npv - AI
lam_base    = (CDS_SPOT_BP / 1e4) / (1.0 - RECOVERY_BASE)

print("\n" + "=" * 66)
print("Q6 -- Credit Valuation Adjustment (CVA)")
print(f"   Trade date : {trade_date}   |   Spot date : {spot_date}")
print("=" * 66)

print(f"\nModel parameters")
print(f"  CDS spread (flat)  : {CDS_SPOT_BP:.2f} bp  (UNIC5YEUAM=R, 05-Nov-2025)")
print(f"  Recovery rate      : {RECOVERY_BASE*100:.0f}%")
print(f"  Hazard rate lambda : {lam_base:.6f} p.a.")
print(f"  Survival prob T40  : {math.exp(-lam_base*tau_40):.4f}  "
      f"(cum. default prob: {(1-math.exp(-lam_base*tau_40))*100:.2f}%)")

print(f"\nCVA breakdown")
print(f"  CVA (coupons)   : EUR {cva_coupons:.4f}  "
      f"({cva_coupons/cva_total*100:.1f}% of total CVA)")
print(f"  CVA (principal) : EUR {cva_principal:.4f}  "
      f"({cva_principal/cva_total*100:.1f}% of total CVA)")
print(f"  CVA (total)     : EUR {cva_total:.4f}  "
      f"({cva_total/rf_npv*100:.2f}% of risk-free NPV)")

print(f"\nValuation summary  (EUR per 1000 nominal)")
print(f"  {'Risk-free NPV (gross)':<30} EUR {rf_npv:>8.4f}  ({rf_npv/10:.4f}%)")
print(f"  {'CVA':<30} EUR {cva_total:>8.4f}  ({cva_total/rf_npv*100:.4f}%)")
print(f"  {'Risky NPV (gross)':<30} EUR {risky_npv:>8.4f}  ({risky_npv/10:.4f}%)")
print(f"  {'Accrued interest':<30} EUR {AI:>8.4f}")
print(f"  {'Risky clean price':<30} EUR {risky_clean:>8.4f}  ({risky_clean/10:.4f}%)")

print("\nPer-period CVA detail:")
print(cva_detail.to_string(index=False))



Risk-free NPV (gross) : EUR 1102.27
Accrued interest      : EUR 4.779
Risk-free clean price : EUR 1097.49  (109.75%)

Q6 -- Credit Valuation Adjustment (CVA)
   Trade date : November 5th, 2025   |   Spot date : November 7th, 2025

Model parameters
  CDS spread (flat)  : 38.78 bp  (UNIC5YEUAM=R, 05-Nov-2025)
  Recovery rate      : 40%
  Hazard rate lambda : 0.006463 p.a.
  Survival prob T40  : 0.9459  (cum. default prob: 5.41%)

CVA breakdown
  CVA (coupons)   : EUR 29.2888  (100.0% of total CVA)
  CVA (principal) : EUR 0.0000  (0.0% of total CVA)
  CVA (total)     : EUR 29.2888  (2.66% of risk-free NPV)

Valuation summary  (EUR per 1000 nominal)
  Risk-free NPV (gross)          EUR 1102.2711  (110.2271%)
  CVA                            EUR  29.2888  (2.6571%)
  Risky NPV (gross)              EUR 1072.9823  (107.2982%)
  Accrued interest               EUR   4.7794
  Risky clean price              EUR 1068.2028  (106.8203%)

Per-period CVA detail:
   Period Payment Date  tau_i     Z_rf

In [6]:
# =============================================================================
# 8.  Save CVA detail and print summary
# =============================================================================

# Enrich the per-period CVA table with forward rate and coupon rate columns
# from the exposure schedule, then save for reference.
cva_out = cva_detail.merge(
    exp_df[["Period", "Payment Date", "Fwd EURIBOR", "Coupon Rate"]],
    on=["Period", "Payment Date"], how="left"
)
cva_out["Fwd EURIBOR %"] = (cva_out["Fwd EURIBOR"] * 100).round(4)
cva_out["Coupon Rate %"] = (cva_out["Coupon Rate"] * 100).round(4)
cva_out = cva_out[[
    "Period", "Payment Date", "tau_i",
    "Fwd EURIBOR %", "Coupon Rate %",
    "Z_rf", "Q(T_{i-1})", "Q(T_i)", "dQ", "CVA_i EUR"
]]

cva_out.to_csv("q6_cva_detail.csv", index=False)
print("Saved: q6_cva_detail.csv")

# =============================================================================
# 9.  Final summary table
# =============================================================================
print("\n" + "=" * 66)
print(f"FINAL SUMMARY  (EUR per 1000 nominal, CDS={CDS_SPOT_BP:.2f}bp, R=40%)")
print("=" * 66)
print(f"{'Item':<35} {'EUR':>10} {'%':>8}")
print("-" * 56)
print(f"{'Risk-free NPV (gross)':<35} {rf_npv:>10.4f} {rf_npv/10:>8.4f}")
print(f"{'CVA – coupons':<35} {cva_coupons:>10.4f} {cva_coupons/rf_npv*100:>8.4f}")
print(f"{'CVA – principal':<35} {cva_principal:>10.4f} {cva_principal/rf_npv*100:>8.4f}")
print(f"{'CVA – total':<35} {cva_total:>10.4f} {cva_total/rf_npv*100:>8.4f}")
print(f"{'Risky NPV (gross)':<35} {risky_npv:>10.4f} {risky_npv/10:>8.4f}")
print(f"{'Accrued interest':<35} {AI:>10.4f}")
print(f"{'Risky clean price':<35} {risky_clean:>10.4f} {risky_clean/10:>8.4f}")


Saved: q6_cva_detail.csv

FINAL SUMMARY  (EUR per 1000 nominal, CDS=38.78bp, R=40%)
Item                                       EUR        %
--------------------------------------------------------
Risk-free NPV (gross)                1102.2711 110.2271
CVA – coupons                          29.2888   2.6571
CVA – principal                         0.0000   0.0000
CVA – total                            29.2888   2.6571
Risky NPV (gross)                    1072.9823 107.2982
Accrued interest                        4.7794
Risky clean price                    1068.2028 106.8203
